# 8.5 Final Model Selection and Deployment — Code Brief

## Key Concepts

- Final recommended model: **Survey-Enhanced XGBoost** (best balance of prediction, actionability, institutional explanation across 8.1-8.4).
- Deployment pipeline: load saved model → predict probability on hold-out set → convert to practical outreach threshold (capacity-based, e.g. top 20%) → risk bands (Priority outreach / Monitor-support / Routine support) → advisor-facing outreach list.
- A deployable model = model artifact + feature schema + model card + deployment checklist, not just the pickle file.
- **Known gap:** `Deploy_Survey_Data.csv` / `Deploy_Data_Other.csv` are not produced by any notebook in this repo — flagged separately, not fixed here.

In [ ]:
# If needed in Colab, uncomment the next line:
# !pip -q install xgboost

import numpy as np
import pandas as pd
import warnings
import time
import joblib
import json
from pathlib import Path

warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, brier_score_loss, log_loss
)

import pickle

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Data Loading Helper

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

# Load training deploy data
df_deploy = pd.read_csv(f'{data_filepath}Deploy_Survey_Data.csv')
pd.set_option('display.max_columns', None)
df_deploy

In [ ]:
# Load training deploy data
df_deploy_names = pd.read_csv(f'{data_filepath}Deploy_Data_Other.csv')
pd.set_option('display.max_columns', None)
df_deploy_names

## Pickle in the Best Model

In [ ]:
# Load the model from the pickle file
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'
filename1 = f'{model_filepath}Survey_xgb_model.pkl'
survey_xgb_model = pickle.load(open(filename1, 'rb'))

filename2 = f'{model_filepath}kmeans_model.pkl'
kmeans_cluster_model = pickle.load(open(filename2, 'rb'))
# You can now use the loaded_model_pkl object for predictions or other tasks
print("Model loaded successfully from pickle file.")

## Deploy the Recommended Model to the Hold-Out Set

In [ ]:
# Predicted probability of departure for each hold-out student
holdout_prob = survey_xgb_model.predict_proba(df_deploy)[:, 1]

# Create an indicator variable that =1 for students predicted to depart and =0 if not
holdout_pred_default = (holdout_prob >= 0.50).astype(int)

holdout_scores = df_deploy.copy()
holdout_scores['departure_risk_score'] = holdout_prob
holdout_scores['predicted_departed_default_050'] = holdout_pred_default

holdout_scores

In [ ]:
# Add a stable row identifier if the dataset does not already include one.
if 'SID' not in holdout_scores.columns and 'STUDENT_ID' not in holdout_scores.columns:
    holdout_scores.insert(0, 'holdout_row_id', np.arange(len(holdout_scores)))

holdout_scores[['departure_risk_score', 'predicted_departed_default_050']].describe().round(4)

## Choose a Practical Outreach Threshold

In [ ]:
capacity_share = 0.20  # Change this if advising capacity is larger or smaller.
capacity_threshold = float(np.quantile(holdout_prob, 1 - capacity_share))
holdout_scores['flag_top_20pct_capacity'] = (holdout_scores['departure_risk_score'] >= capacity_threshold).astype(int)

print(f'Capacity-based threshold for top {capacity_share:.0%}: {capacity_threshold:.4f}')
print('Number flagged:', int(holdout_scores['flag_top_20pct_capacity'].sum()))
print('Share flagged:', holdout_scores['flag_top_20pct_capacity'].mean().round(4))


## Convert Risk Scores into Stakeholder-Friendly Risk Bands

In [ ]:
def assign_risk_band(score):
    if score >= np.quantile(holdout_prob, 0.80):
        return 'Priority outreach'
    elif score >= np.quantile(holdout_prob, 0.50):
        return 'Monitor/support'
    else:
        return 'Routine support'

holdout_scores['support_band'] = holdout_scores['departure_risk_score'].apply(assign_risk_band)

band_summary = (
    holdout_scores
    .groupby('support_band')
    .agg(
        students=('departure_risk_score', 'size'),
        avg_risk_score=('departure_risk_score', 'mean'),
    )
    .sort_values('avg_risk_score', ascending=False)
)

band_summary.round(4)

In [ ]:
fig = px.bar(
    band_summary.reset_index(),
    x='support_band',
    y='students',
    text='students',
    title='Hold-Out Students by Support Band',
    labels={'support_band': 'Support Band', 'students': 'Number of Students'}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450)
fig.show()

## Create an Advisor-Facing Outreach List

In [ ]:
df_deploy_risk = pd.concat([df_deploy_names[['SID','NAME','LAST_NAME']],holdout_scores],axis=1)

In [ ]:

recommended_context_cols = [
    'SID',
    'NAME',
    'LAST_NAME',
    'departure_risk_score',
    'support_band',
    'flag_top_20pct_capacity',
    'HS_GPA',
    'GPA_1',
    'GPA_2',
    'DFW_RATE_1',
    'DFW_RATE_2',
    'UNITS_ATTEMPTED_1',
    'UNITS_ATTEMPTED_2'
]



advisor_outreach_list = (
    df_deploy_risk[recommended_context_cols]
    .sort_values('departure_risk_score', ascending=False)
    .reset_index(drop=True)
)

advisor_outreach_list.head(123).round(4)

## Model Card for Stakeholders

| Field | Description |
|:------|:-----------|
| **Model Name** | Survey-Enhanced Student Departure Risk Model v1.0 |
| **Model Type** | XGBoost binary classifier |
| **Task** | Binary classification: predict 3rd semester departure |
| **Performance (AUC)** | 0.8791 |
| **Intended Use** | Early warning system for academic advisors |
| **Limitations** | Trained on CSULB data; may not generalize to other institutions |

## Deployment Checklist

Technical validation, hold-out performance, threshold choice, equity review, privacy review, documentation, human oversight, monitoring, governance — see full checklist in the lecture.